In [1]:
import torch
import esm

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [20]:
#model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
#model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
model, alphabet = esm.pretrained.esm2_t30_150M_UR50D()
last_layer = len(model.layers) #get encoder layers
batch_converter = alphabet.get_batch_converter() #tokenise

model.eval() #evaluation model
model.to(device)

alphabet.to_dict()


Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t30_150M_UR50D.pt" to /home/nisanga/.cache/torch/hub/checkpoints/esm2_t30_150M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t30_150M_UR50D-contact-regression.pt" to /home/nisanga/.cache/torch/hub/checkpoints/esm2_t30_150M_UR50D-contact-regression.pt


{'<cls>': 0,
 '<pad>': 1,
 '<eos>': 2,
 '<unk>': 3,
 'L': 4,
 'A': 5,
 'G': 6,
 'V': 7,
 'S': 8,
 'E': 9,
 'R': 10,
 'T': 11,
 'I': 12,
 'D': 13,
 'P': 14,
 'K': 15,
 'Q': 16,
 'N': 17,
 'F': 18,
 'Y': 19,
 'M': 20,
 'H': 21,
 'W': 22,
 'C': 23,
 'X': 24,
 'B': 25,
 'U': 26,
 'Z': 27,
 'O': 28,
 '.': 29,
 '-': 30,
 '<null_1>': 31,
 '<mask>': 32}

In [21]:
def get_protein_embedding(amino_acid_seq, name):

    batch = [(name, amino_acid_seq)]

    batch_labels, batch_strs, batch_tokens = batch_converter(batch)
    
    batch_tokens = batch_tokens.to(device)

    with torch.no_grad():
        esm2_result = model(batch_tokens, repr_layers=[last_layer], return_contacts=False)

    token_representations = esm2_result["representations"][last_layer]

    # exlude start and end token
    token_representations = token_representations[0, 1:-1, :]

    return token_representations


In [22]:
pcv_seq = "MPSKKNGRSGPQPHKRWVFTLNNPSEDERKKIREPPISLFDYFIVGEEGNEEGRTPHLQGFANFVKKQTFNKVKWYLGARCHIEKAKGTDQQNKEYCSKEGNLLIECGAPRSQGQRSDLSTAVSTLLESGSLVTVAEQHPVTFVRNFRGLAELLKVSGKMQKRDWKTNVHVIVGPPGCGKSKWAANFADPETTYWKPPRNKWWDGYHGEEVVVIDDFYGWLPWDDLLRLCDRYPLTVETKGGTVPFLARSILITSNQTPLEWYSSTAVPAVEALYRRITSLVFWKNATEQSTEEGGQFVTLSPPCPEFPYEINY"

wdv_seq = "MASSSAPRFRVYSKYLFLTYPECTLEPQYALDSLRTLLNKYEPLYIAAVRELHEDGSPHLHVLVQNKLRASITNPNALNLRMDTSPFSIFHPNIQAAKDCNQVRDYITKEVDSDVNTAEWGTFVAVSTPGRKDRDADMKQIIESSSSREEFLSMVCNRFPFEWSIRLKDFEYTARHLFPDPVATYTPEFPTESLICHETIESWKNEHLYSESPGRHKSIYICGPTRTGKTSWARSLGTHNYYNSLVDFTTYDVNAKYNIIDDIPFKFTPNWKCFVGAQRDFTVNPKYGKRKVIRGGIPCIILVNPDEDWLKDMTPEQSDYMYSNTVVHYMYEGETFINYSFASGEDVTASQ"

wdv_embed = get_protein_embedding(wdv_seq, "WDV Rep")

pcv_embed = get_protein_embedding(pcv_seq, "PCV Rep")

print(wdv_embed.shape)
print(pcv_embed.shape)

torch.Size([351, 640])
torch.Size([314, 640])


In [23]:
wdv_sDBM = wdv_embed[90:99, :]

wdv_sDBM.shape

torch.Size([9, 640])

In [24]:
import torch.nn.functional as F

In [36]:
length, _ = pcv_embed.shape

count = 0

max_mean = -1

start_seq = 0
end_seq = 0

for i in range(length - 8):
    block = pcv_embed[i:i+9, :]

    sim = F.cosine_similarity(block, wdv_sDBM, dim = 1)

    mean = torch.mean(sim)

    if mean > max_mean:
        start_seq = i + 1
        end_seq = i + 9
        max_mean = mean


print(start_seq, end_seq)

81 89


In [25]:
fbnyv_seq = "MARQVICWCFTLNNPLSPLSLHDSMKYLVYQTEQGEAGNIHFQGYIEMKKRTSLAGMKKLIPGAHFEKRRGTQGEARAYSMKEDTRLEGPWEYGEFVPTIEDKLREVMNDMKITGKRPIEYIEECCNTYDKSASTLREFRGELKKKKAISSWELQRKPWMGEVDALLQERDGRRIIWVYGPQGGEGKTSYAKHLVKTRDAFYSTGGKTADIAFAWDHQELVLFDFPRSFEEYVNYGVIEQLKNGIIQSGKYQSVIKYSDYVEVIVFANFTPRSGMFSEDRIVYVYA"

fbnyv_embed = get_protein_embedding(fbnyv_seq, "FBNYV Rep")

fbnyv_embed.shape

torch.Size([286, 640])

In [26]:
dcv_seq = "MAKSGNYSYKRWVFTINNPTFEDYVHVLEFCTLDNCKFAIVGEEKGANGTPHLQGFLNLRSNARAAALEESLGGRAWLSRARGSDEDNEEYCAKESTYLRVGEPVSKGRSSDLAEATSAVMAGVPLTEVARKFPTTYVIFGRGLERLRHLIVETQRDWKTEVIVLIGPPGTGKSRYAFEFPAENKYYKPRGKWWDGYSGNDVVVMDDFYGWLPYDDLLRITDRCPLRVEFKGGMTQFVAKTLIITSNREPRDWYKSEFDLSALYRRINKYLVYNIDKYEPAQACTLPFPINY"

dcv_embed = get_protein_embedding(dcv_seq, "DCV Rep")

In [27]:
pcv_sDBM = pcv_embed[78:87, :]

pcv_sDBM.shape

torch.Size([9, 640])

In [28]:
def query_sDBM(protein_seq, known_sDBM):

    protein_embed = get_protein_embedding(protein_seq, "Rep Protein")

    length, _ = protein_embed.shape

    max_mean = -1
    next_max_mean = -1

    start_seq = 0
    end_seq = 0 

    for i in range(length - 8):
        match_block = protein_embed[i:i+9, :]

        sim = F.cosine_similarity(match_block, known_sDBM, dim = 1)

        mean = torch.mean(sim)

        # print(i + 1)
        # print(mean)

        if mean > max_mean:
            start_seq = i + 1
            end_seq = i + 9
            next_max_mean = max_mean
            max_mean = mean
        elif mean > next_max_mean:
            next_max_mean = mean

    if ((max_mean - next_max_mean) < 0.01):
        print("Unable to suggest sDBM... confidence measure below threshold")
        return


    print("sDBM candidate found!!!")
    print(f"Max Mean Similarity: {max_mean}")
    print(f"Closest Mean Similarity to sDBM Prediction: {next_max_mean}")
    print(f"sDBM prediction from {start_seq} to {end_seq}")
    print(protein_seq[start_seq - 1: end_seq])

    #return (max_mean, (start_seq, end_seq))



In [29]:
tgmv_seq = "MPSHPKRFQINAKNYFLTYPQCSLSKEESLSQLQALNTPINKKFIKICRELHEDGQPHLHVLIQFEGKYCCQNQRFFDLVSPTRSAHFHPNIQRAKSSSDVKTYIDKDGDTLVWGEFQVDGRSARGGCQTSNDAAAEALNASSKEEALQIIREKIPEKYLFQFHNLNSNLDRIFDKTPEPWLPPFHVSSFTNVPDEMRQWAENYFGKSSAARPERPISIIIEGDSRTGKTMWARSLGPHNYLSGHLDLNSRVYSNKVEYNVIDDVTPQYLKLKHWKELIGAQRDWQTNCKYGKPVQIKGGIPSIVLCNPGEGASYKVFLDKEENTPLKNWTFHNAKFVFLNSPLYQSSTQSS"

query_sDBM(tgmv_seq, wdv_sDBM)

print(f"Query sDBM: {wdv_seq[90:99]}")

sDBM candidate found!!!
Max Mean Similarity: 0.9416740536689758
Closest Mean Similarity to sDBM Prediction: 0.691121518611908
sDBM prediction from 89 to 97
HPNIQRAKS
Query sDBM: HPNIQAAKD


In [102]:
pcv_sDBM = pcv_embed[79:88, :]

print(pcv_seq[79:88])

query_sDBM(dcv_seq, pcv_sDBM)



RCHIEKAKG
Max Mean Similarity: 0.8672306537628174
sDBM prediction from 75 to 83
RAWLSRARG


In [109]:
query_sDBM(wdv_seq, pcv_sDBM)
query_sDBM(pcv_seq, wdv_sDBM)

Max Mean Similarity: 0.8020769357681274
sDBM prediction from 175 to 183
RHLFPDPVA
Max Mean Similarity: 0.7955700755119324
sDBM prediction from 80 to 88
RCHIEKAKG


In [37]:
# Does not do the search beyond 150


def query_sDBM_restricted(protein_seq, known_sDBM):

    protein_embed = get_protein_embedding(protein_seq, "Rep Protein")

    length, _ = protein_embed.shape

    max_mean = -1
    next_max_mean = -1

    start_seq = 0
    end_seq = 0 

    for i in range(30, 150):
        match_block = protein_embed[i:i+9, :]

        sim = F.cosine_similarity(match_block, known_sDBM, dim = 1)

        mean = torch.mean(sim)

        # print(i + 1)
        # print(mean)

        if mean > max_mean:
            start_seq = i + 1
            end_seq = i + 9
            next_max_mean = max_mean
            max_mean = mean
        elif mean > next_max_mean:
            next_max_mean = mean

    if ((max_mean - next_max_mean) < 0.01):
        print("Unable to suggest sDBM... confidence measure below threshold")
        return


    print("sDBM candidate found!!!")
    print(f"Max Mean Similarity: {max_mean}")
    print(f"Closest Mean Similarity to sDBM Prediction: {next_max_mean}")
    print(f"sDBM prediction from {start_seq} to {end_seq}")
    print(protein_seq[start_seq - 1: end_seq])

    #return (max_mean, (start_seq, end_seq))

In [33]:
bbtv_seq = "MARYVVCWMFTINNPASLPVMREEFKYMVYQVERGQEGTRHVQGYVEMKRRSSLKQMRGFFPGAHLEKRKGSQEEARAYCMKEDTRIEGPFEFGAFKLSCNDNLFDVIQDMRETHKRPLEYLYECPNTFDRSKDTLYRVQAELNKTKAMNSWKTSFSSWTSEVENIMAEPCHRRIIWVYGPNGGEGKTTYAKYLMKTKNAFYSPGGKSLDICRLYNYEEIVIFDIPRCKEEYLNYGLLEEFKNGIIQSGKYEPVLKIVEYVEVIVMANFLPKEGIFSEDRIKLVAC"

# finding sDBM quickly with index:
# print(bbtv_seq.index("GAHLE"))

# print(fbnyv_seq.index("GAHFE"))


bbtv_sDBM = get_protein_embedding(bbtv_seq, "Rep")[62:71, :]
fbnyv_sDBM = get_protein_embedding(fbnyv_seq, "Rep")[62:71, :]

query_sDBM(fbnyv_seq, bbtv_sDBM)
query_sDBM(fbnyv_seq, pcv_sDBM)
query_sDBM(fbnyv_seq, wdv_sDBM)

print("----")

query_sDBM(dcv_seq, pcv_sDBM)
query_sDBM(dcv_seq, wdv_sDBM)
query_sDBM(dcv_seq, bbtv_sDBM)

print("----")

query_sDBM(tgmv_seq, wdv_sDBM)
query_sDBM(tgmv_seq, pcv_sDBM)
query_sDBM(tgmv_seq, fbnyv_sDBM)


sDBM candidate found!!!
Max Mean Similarity: 0.9685831069946289
Closest Mean Similarity to sDBM Prediction: 0.6183121800422668
sDBM prediction from 63 to 71
GAHFEKRRG
sDBM candidate found!!!
Max Mean Similarity: 0.8558032512664795
Closest Mean Similarity to sDBM Prediction: 0.6496512293815613
sDBM prediction from 62 to 70
PGAHFEKRR
sDBM candidate found!!!
Max Mean Similarity: 0.7124722003936768
Closest Mean Similarity to sDBM Prediction: 0.6764393448829651
sDBM prediction from 63 to 71
GAHFEKRRG
----
sDBM candidate found!!!
Max Mean Similarity: 0.8898860216140747
Closest Mean Similarity to sDBM Prediction: 0.6448941230773926
sDBM prediction from 74 to 82
GRAWLSRAR
sDBM candidate found!!!
Max Mean Similarity: 0.7594119310379028
Closest Mean Similarity to sDBM Prediction: 0.6905804872512817
sDBM prediction from 75 to 83
RAWLSRARG
sDBM candidate found!!!
Max Mean Similarity: 0.8630154728889465
Closest Mean Similarity to sDBM Prediction: 0.6706759333610535
sDBM prediction from 75 to 83
RAW

In [36]:

query_sDBM_restricted(fbnyv_seq, bbtv_sDBM)
query_sDBM_restricted(fbnyv_seq, pcv_sDBM)
query_sDBM_restricted(fbnyv_seq, wdv_sDBM)

print("----")

query_sDBM_restricted(dcv_seq, pcv_sDBM)
query_sDBM_restricted(dcv_seq, wdv_sDBM)
query_sDBM_restricted(dcv_seq, bbtv_sDBM)

print("----")

query_sDBM_restricted(tgmv_seq, wdv_sDBM)
query_sDBM_restricted(tgmv_seq, pcv_sDBM)
query_sDBM_restricted(tgmv_seq, fbnyv_sDBM)

sDBM candidate found!!!
Max Mean Similarity: 0.9685831069946289
Closest Mean Similarity to sDBM Prediction: 0.6183121800422668
sDBM prediction from 63 to 71
GAHFEKRRG
sDBM candidate found!!!
Max Mean Similarity: 0.8558032512664795
Closest Mean Similarity to sDBM Prediction: 0.6496512293815613
sDBM prediction from 62 to 70
PGAHFEKRR
sDBM candidate found!!!
Max Mean Similarity: 0.7124722003936768
Closest Mean Similarity to sDBM Prediction: 0.6764393448829651
sDBM prediction from 63 to 71
GAHFEKRRG
----
sDBM candidate found!!!
Max Mean Similarity: 0.8898860216140747
Closest Mean Similarity to sDBM Prediction: 0.6448941230773926
sDBM prediction from 74 to 82
GRAWLSRAR
sDBM candidate found!!!
Max Mean Similarity: 0.7594119310379028
Closest Mean Similarity to sDBM Prediction: 0.6905804872512817
sDBM prediction from 75 to 83
RAWLSRARG
sDBM candidate found!!!
Max Mean Similarity: 0.8630154728889465
Closest Mean Similarity to sDBM Prediction: 0.6706759333610535
sDBM prediction from 75 to 83
RAW